# YZTA 5.0 Datathon — Bilişsel Performans Skoru Tahmini

Bu çalışma, verilen kişi/uyku/yaşam tarzı değişkenlerinden `bilissel_performans_skoru` değerini tahmin etmek için hazırlanmıştır.

**Problem tipi:** Regresyon  
**Değerlendirme metriği:** RMSE  
**Model:** CatBoost Regressor  
**Public leaderboard skoru:** 1.20556

Çalışma akışı:

1. Veri yükleme ve temel kontroller
2. Eksik değer ve değişken tipi incelemesi
3. Feature engineering
4. K-Fold cross validation
5. Final test tahminleri
6. Kaggle submission dosyası üretimi

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

RANDOM_STATE = 42
TARGET = "bilissel_performans_skoru"
N_SPLITS = 3

## 1. Veri dosyalarının yüklenmesi

Notebook hem lokal klasörde hem de Kaggle ortamında çalışabilecek şekilde dosya yollarını otomatik arar.
Beklenen dosyalar:

- `train.csv`
- `test_x.csv`
- `sample_submission.csv`

In [ ]:
def find_file(filename):
    candidate_paths = [
        filename,
        f"./{filename}",
        f"../input/{filename}",
        f"/kaggle/input/{filename}",
    ]

    for path in candidate_paths:
        if os.path.exists(path):
            return path

    # Kaggle input klasörlerinde alt klasör araması
    kaggle_root = "/kaggle/input"
    if os.path.exists(kaggle_root):
        for root, _, files in os.walk(kaggle_root):
            if filename in files:
                return os.path.join(root, filename)

    raise FileNotFoundError(f"Dosya bulunamadı: {filename}")

train_path = find_file("train.csv")
test_path = find_file("test_x.csv")
sample_path = find_file("sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("train:", train.shape)
print("test:", test.shape)
print("sample_submission:", sample_submission.shape)

## 2. Temel veri kontrolleri

Bu bölümde kolonlar, hedef değişken ve eksik değerler kontrol edilir.

In [ ]:
print("Train kolonları:")
print(train.columns.tolist())

print("
Target özeti:")
print(train[TARGET].describe())

missing_summary = pd.DataFrame({
    "train_missing_count": train.isna().sum(),
    "train_missing_ratio": train.isna().mean(),
    "test_missing_count": test.isna().sum(),
    "test_missing_ratio": test.isna().mean(),
}).sort_values("train_missing_ratio", ascending=False)

missing_summary[missing_summary.sum(axis=1) > 0]

In [ ]:
# Sayısal ve kategorik kolonları hızlı kontrol
base_features = [c for c in train.columns if c not in ["id", TARGET]]
numeric_cols = train[base_features].select_dtypes(exclude="object").columns.tolist()
categorical_cols = train[base_features].select_dtypes(include="object").columns.tolist()

print("Sayısal kolon sayısı:", len(numeric_cols))
print(numeric_cols)
print("
Kategorik kolon sayısı:", len(categorical_cols))
print(categorical_cols)

## 3. Feature engineering

Bu bölümde mevcut değişkenlerden model performansını artırabilecek yeni değişkenler oluşturulur.
Temel yaklaşım:

- Uyku kalitesiyle ilgili birleşik göstergeler
- Stres, çalışma ve sağlık etkileşimleri
- Kafein/ekran süresi davranış değişkenleri
- Hafta sonu ve ortam sıcaklığı göstergeleri
- Kategorik değişken etkileşimleri
- Eksik değer bayrakları

In [ ]:
def add_features(df):
    df = df.copy()
    eps = 1e-6

    # Train/test arasında görülebilecek kategori yazım farklarını azaltma
    df["ulke_norm"] = df["ulke"].replace({
        "South Korea": "Guney Kore",
        "Spain": "Ispanya",
        "Sweden": "Isvec",
        "Mexico": "Meksika",
        "Netherlands": "Hollanda",
    })
    df["meslek_norm"] = df["meslek"].replace({"Lawyer": "Avukat"})

    # Uyku kalitesi göstergeleri
    df["rem_derin_toplam"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]
    df["hafif_uyku_yuzdesi"] = 100 - df["rem_yuzdesi"] - df["derin_uyku_yuzdesi"]
    df["rem_derin_oran"] = df["rem_yuzdesi"] / (df["derin_uyku_yuzdesi"] + eps)
    df["derin_rem_oran"] = df["derin_uyku_yuzdesi"] / (df["rem_yuzdesi"] + eps)
    df["uyku_kalitesi_raw"] = 0.45 * df["rem_yuzdesi"] + 0.55 * df["derin_uyku_yuzdesi"]
    df["uyku_bolunme_skoru"] = df["uykuya_dalma_suresi_dk"] + 10 * df["gecelik_uyanma_sayisi"]
    df["iyi_uyku_indeksi"] = (
        df["rem_yuzdesi"]
        + df["derin_uyku_yuzdesi"]
        - 0.25 * df["uykuya_dalma_suresi_dk"]
        - 2 * df["gecelik_uyanma_sayisi"]
    )
    df["uyku_verimlilik_proxy"] = (
        df["rem_derin_toplam"]
        - 0.10 * df["hafif_uyku_yuzdesi"]
        - 2 * df["gecelik_uyanma_sayisi"]
    )

    # Davranış göstergeleri
    df["kafein_ekran_toplam"] = df["uyku_oncesi_kafein_mg"] / 50 + df["uyku_oncesi_ekran_suresi_dk"] / 60
    df["kafein_ekran_carpim"] = df["uyku_oncesi_kafein_mg"] * df["uyku_oncesi_ekran_suresi_dk"]
    df["aktivite_calisma_oran"] = df["gunluk_adim_sayisi"] / (df["gunluk_calisma_saati"] + 1)
    df["adim_bin_proxy"] = df["gunluk_adim_sayisi"] / 1000
    df["sekerleme_saat"] = df["sekerleme_suresi_dk"] / 60

    # Stres, sağlık ve çalışma etkileşimleri
    df["stres_calisma"] = df["stres_skoru"] * df["gunluk_calisma_saati"]
    df["stres_uyku_bolunme"] = df["stres_skoru"] * df["gecelik_uyanma_sayisi"]
    df["stres_kafein"] = df["stres_skoru"] * df["uyku_oncesi_kafein_mg"]
    df["stres_ekran"] = df["stres_skoru"] * df["uyku_oncesi_ekran_suresi_dk"]
    df["nabiz_stres"] = df["dinlenik_nabiz_bpm"] * df["stres_skoru"]
    df["saglik_risk_indeksi"] = df["stres_skoru"] + df["vucut_kitle_indeksi"] / 10 + df["dinlenik_nabiz_bpm"] / 20
    df["bmi_nabiz"] = df["vucut_kitle_indeksi"] * df["dinlenik_nabiz_bpm"]
    df["yas_stres"] = df["yas"] * df["stres_skoru"]
    df["yas_nabiz"] = df["yas"] * df["dinlenik_nabiz_bpm"]

    # Zaman / ortam göstergeleri
    df["hafta_sonu_mu"] = (df["gun_tipi"] == "Hafta sonu").astype(int)
    df["sicaklik_idealden_sapma"] = (df["oda_sicakligi_celsius"] - 20).abs()
    df["sicaklik_idealden_sapma2"] = df["sicaklik_idealden_sapma"] ** 2
    df["hafta_sonu_farki_abs"] = df["hafta_sonu_uyku_farki_saat"].abs()
    df["hafta_sonu_farki2"] = df["hafta_sonu_uyku_farki_saat"] ** 2

    # Seçilmiş değişkenlerde kare/log dönüşümleri
    for col in [
        "stres_skoru",
        "uyku_oncesi_ekran_suresi_dk",
        "uyku_oncesi_kafein_mg",
        "gunluk_calisma_saati",
        "gecelik_uyanma_sayisi",
        "uykuya_dalma_suresi_dk",
    ]:
        df[f"{col}_sq"] = df[col] ** 2
        df[f"{col}_log1p"] = np.log1p(df[col])

    # Kategorik etkileşimler
    df["kronotip_gun_tipi"] = df["kronotip"].astype(str) + "_" + df["gun_tipi"].astype(str)
    df["ruh_kronotip"] = df["ruh_sagligi_durumu"].astype(str) + "_" + df["kronotip"].astype(str)
    df["meslek_gun_tipi"] = df["meslek_norm"].astype(str) + "_" + df["gun_tipi"].astype(str)
    df["ulke_mevsim"] = df["ulke_norm"].astype(str) + "_" + df["mevsim"].astype(str)
    df["cinsiyet_kronotip"] = df["cinsiyet"].astype(str) + "_" + df["kronotip"].astype(str)

    # Eksik değer bayrakları
    for col in ["meslek", "vucut_kitle_indeksi", "uyku_oncesi_kafein_mg", "stres_skoru", "kronotip", "ruh_sagligi_durumu"]:
        df[f"{col}_missing"] = df[col].isna().astype(int)

    return df

train_fe = add_features(train)
test_fe = add_features(test)

print("Feature engineering sonrası train:", train_fe.shape)
print("Feature engineering sonrası test:", test_fe.shape)

## 4. Model matrisi hazırlığı

`id` ve hedef değişken model girdisinden çıkarılır. CatBoost kategorik değişkenleri doğrudan işleyebildiği için kategorik kolon indeksleri ayrıca verilir.

In [ ]:
features = [col for col in train_fe.columns if col not in ["id", TARGET]]

X = train_fe[features].copy()
y = train_fe[TARGET].copy()
X_test = test_fe[features].copy()

cat_cols = X.select_dtypes(include="object").columns.tolist()

for col in cat_cols:
    X[col] = X[col].astype(str).fillna("__MISSING__")
    X_test[col] = X_test[col].astype(str).fillna("__MISSING__")

cat_features = [X.columns.get_loc(col) for col in cat_cols]

print("Feature sayısı:", len(features))
print("Kategorik feature sayısı:", len(cat_cols))

## 5. Cross validation ve model eğitimi

Model kalitesi `KFold` cross validation ile ölçülür. Her fold'da model farklı bir validasyon bölümünde test edilir. Nihai CV skoru, tüm validasyon tahminlerinin RMSE değeridir.

In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []
models = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostRegressor(
        iterations=1600,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=RANDOM_STATE + fold,
        od_type="Iter",
        od_wait=100,
        verbose=200,
        allow_writing_files=False,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
    )

    valid_pred = np.clip(model.predict(X_valid), 0, 10)
    test_pred = np.clip(model.predict(X_test), 0, 10)

    oof_preds[valid_idx] = valid_pred
    test_preds += test_pred / N_SPLITS
    models.append(model)

    fold_rmse = rmse(y_valid, valid_pred)
    fold_scores.append(fold_rmse)
    print(f"Fold {fold} RMSE: {fold_rmse:.5f}")

cv_rmse = rmse(y, oof_preds)
print("Fold skorları:", [round(score, 5) for score in fold_scores])
print(f"Genel CV RMSE: {cv_rmse:.5f}")

## 6. Feature importance

CatBoost'un değişken önem skorları incelenerek modelin hangi değişkenlerden daha fazla yararlandığı görülebilir.

In [ ]:
feature_importance = np.mean(
    [model.get_feature_importance() for model in models],
    axis=0,
)

importance_df = pd.DataFrame({
    "feature": features,
    "importance": feature_importance,
}).sort_values("importance", ascending=False)

importance_df.head(25)

## 7. Submission dosyası üretimi

Kaggle'a yüklenecek dosya `sample_submission.csv` formatıyla aynı kolonlara sahip olmalıdır.

In [ ]:
submission = sample_submission.copy()
submission[TARGET] = np.clip(test_preds, 0, 10)

assert list(submission.columns) == ["id", TARGET]
assert len(submission) == len(test)

submission_path = "catboost_submission.csv"
submission.to_csv(submission_path, index=False)

importance_df.to_csv("feature_importance.csv", index=False)

print("Submission dosyası:", submission_path)
print("Satır sayısı:", len(submission))
submission.head()

## 8. Deney kayıt tablosu

| Deneme | Model | Değişiklik | Public Score |
|---|---|---|---:|
| 1 | XGBoost | Baseline + feature engineering | 1.21126 |
| 2 | CatBoost | Kategorik değişken desteği + feature engineering | 1.20556 |

Not: Public leaderboard skoru nihai sonuç değildir. Private leaderboard ayrımı nedeniyle final seçiminde sadece public skora göre karar verilmemelidir.